# 개인실습 — 한빛은행 창구 운영 데이터

> 오늘 배운 **String·TTL·Hash·List**를 한빛은행 창구 운영 상황에 그대로 붙여 봅니다. `db-redis` 컨테이너가 실행 중이어야 해요(`NoSQL_Redis_4_기본_자료구조.ipynb`의 연결 셀과 같은 방식으로 접속합니다).
>
> 빈칸 12자리(명령 10종) — 💡 힌트 먼저, 그다음 📝 정답. 진짜 목표는 **데이터에 맞는 자료구조를 스스로 고르는 것**입니다.


In [1]:
# ✅ 연결 준비 — NoSQL_Redis_4_기본_자료구조.ipynb와 동일
import redis, time

r = redis.Redis(host="localhost", port=6379, db=0, decode_responses=True)
print("PING ->", r.ping())


PING -> True


## 실습 1 — 일회용 인증번호 (String + TTL)

창구에서 본인확인용 일회용 인증번호(OTP)를 발급합니다. 고객 `cust:2001`에게 인증번호 `483920`을 발급하고, **3분(180초) 뒤 자동 소멸**되게 하세요.

💡 힌트: `r.set(key, value, ex=초)` → `r.ttl(key)` → `r.get(key)` (`setex()`는 deprecated!)
👇 아래 빈 셀에 직접 작성해 보세요 (빈칸 4개: 저장·TTL 값·TTL 조회·값 조회)

In [6]:
# 🧪 실습 1 — key="auth:cust:2001", otp="483920", 유효시간 3분(180초)
r.set("auth:cust:2001", 483920, ex=180)

True

In [9]:
#따로 분리하면 누를 때마다 시간이 갱신되는 것 확인 가능
print("남은 시간: ", r.ttl("auth:cust:2001"), "초")
print("인증번호: ", r.get("auth:cust:2001"))

남은 시간:  163 초
인증번호:  483920


## 실습 2 — 창구 상담 카드 (Hash)

상담원이 고객 `cust:2001`의 상담 카드를 작성합니다: 이름 "이서준", 방문목적 "계좌개설", 방문횟수 1회. 상담이 길어져 방문횟수를 다시 세야 한다면(창구를 두 번 오간 경우) 방문횟수를 1 올리세요.

💡 힌트: `r.hset(key, mapping={...})` → `r.hincrby(key, field, 1)` → `r.hgetall(key)`
👇 아래 빈 셀에 직접 작성해 보세요 (빈칸 4개: Hash 저장·방문횟수 증가·전체 조회·필드 하나만 조회)

In [ ]:
# 🧪 실습 2 — key="counter:cust:2001"
r.delete("counter:cust:2001")
r.hset("counter:cust:2001", mapping={"이름": "이서준", "방문목적": "계좌개설", "방문횟수": "1"})
r.hincrby("counter:cust:2001", "방문횟수", 1)
r.hgetall("counter:cust:2001")

{'이름': '이서준', '방문목적': '계좌개설', '방문횟수': '2'}

## 실습 3 — 대기 번호표 (List, FIFO)

번호표 101, 102, 103을 순서대로 발급했습니다. 창구가 비면 **먼저 발급된 번호부터** 호출해야 합니다.

💡 힌트: `LPUSH`(왼쪽 삽입) + `BRPOP`(오른쪽에서 꺼냄) = FIFO(`NoSQL_Redis_4_기본_자료구조.ipynb`의 List와 같은 방향)
👇 아래 빈 셀에 직접 작성해 보세요 (빈칸 4개: 번호표 발급 3회·대기인원 확인·다음 호출·남은 대기열 조회)


In [17]:
# 🧪 실습 3 — key="waiting:branch1", 번호표 101 -> 102 -> 103 순서로 발급
r.delete("waiting:branch1")
r.lpush("waiting:branch1", "101", "102", "103")
print("창구 비우는 순서: ", r.lrange("waiting:branch1", 0, -1)) 
print("맨 마지막에 나오는 것(처음에 넣은 것): ", r.brpop("waiting:branch1", timeout=1))

창구 비우는 순서:  ['103', '102', '101']
맨 마지막에 나오는 것(처음에 넣은 것):  ('waiting:branch1', '101')


## 정답 확인

세 실습을 모두 풀어봤다면 `NoSQL_Redis_5_한빛은행_창구_운영_데이터_개인실습_정답.ipynb`에서 정답과, 정답을 이어서 실행했을 때의 실제 결과를 확인하세요.
